# 09｜Model Explainability & Nested OOF Error Analysis

08 已經完成模型選擇和 Nested Leave-One-Lot-Out 預測，所以 09 不再重新挑模型。這一本我想回答的是：**模型到底靠哪些製程資訊做判斷、這些依賴在不同 Lot 間穩不穩，以及模型在哪些 Lot / wafer 最容易失準。**

這裡會從四個角度看：

1. Final model 的係數主要集中在哪些 engineered features / sensors。
2. 固定同一個 final configuration，輪流拿掉不同 Lot 後，主要係數方向是否還一致。
3. 用真正的 nested outer models 做 sensor-group permutation，確認 held-out Lot prediction 對哪些 sensor groups 最敏感。
4. 從 Nested OOF error 往下追，看看最難預測的 Lot 和 wafer 到底發生什麼事。

這一份的重點是**解釋模型怎麼用資料**，不是把 feature importance 當成製程因果。某個 sensor 對模型很重要，不代表調整它就一定會讓 etch 結果照同一個方向改變。

## 1. 先把 08 的 preprocessing 和 model builder 原樣準備好

09 需要重建 08 當時每一個 outer fold 的模型，所以 near-constant filter、StandardScaler 和 model 參數不能自己換規則。這一步只是在準備相同的建模環境，確保後面的 explainability 是接著 08 做，而不是另外開一條新的模型流程。

In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_rows", 200)

def find_project_root(start=None):
    """Find the project root by walking upward from the current directory."""
    start = Path.cwd().resolve() if start is None else Path(start).resolve()

    for candidate in [start, *start.parents]:
        if (
            (candidate / "notebooks").exists()
            and (candidate / "data").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Project root not found. Expected a parent directory "
        "containing both notebooks/ and data/."
    )


CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = find_project_root(CURRENT_DIR)


PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

class RelativeToleranceFeatureFilter(BaseEstimator, TransformerMixin):
    """沿用 08 的 fold-local near-constant 判定規則。"""
    def __init__(self, rtol=1e-7, atol=1e-8):
        self.rtol = rtol
        self.atol = atol
    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        if X.ndim != 2 or X.shape[0] == 0 or X.shape[1] == 0:
            raise ValueError("X 必須是非空 2D matrix")
        if not np.isfinite(X).all():
            raise ValueError("Near-constant filter 收到 NaN / Inf")
        reference = X[0]
        tolerance = self.atol + self.rtol * np.abs(reference)
        max_deviation = np.max(np.abs(X - reference), axis=0)
        self.support_ = max_deviation > tolerance
        self.n_features_in_ = X.shape[1]
        if not self.support_.any():
            raise ValueError("所有 predictors 都被判定為 near-constant")
        return self
    def transform(self, X):
        return np.asarray(X, dtype=float)[:, self.support_]
    def get_support(self, indices=False):
        if not hasattr(self, "support_"):
            raise ValueError("Filter 尚未 fit")
        return np.flatnonzero(self.support_) if indices else self.support_.copy()

def build_estimator(model_name, params):
    steps = [
        ("variance", RelativeToleranceFeatureFilter(rtol=1e-7, atol=1e-8)),
        ("scale", StandardScaler()),
    ]
    if model_name == "Ridge":
        model = Ridge(alpha=params["alpha"])
    elif model_name == "ElasticNet":
        model = ElasticNet(
            alpha=params["alpha"],
            l1_ratio=params["l1_ratio"],
            max_iter=20_000,
            tol=1e-4,
            random_state=42,
        )
    elif model_name == "PLS":
        model = PLSRegression(
            n_components=params["n_components"],
            scale=False,
            max_iter=500,
        )
    elif model_name == "SVR":
        model = SVR(
            kernel="rbf",
            C=params["C"],
            gamma=params["gamma"],
            epsilon=0.05,
        )
    else:
        raise ValueError(f"未知 model：{model_name}")
    return Pipeline(steps + [("model", model)])

print("模型解釋環境已準備完成")


模型解釋環境已準備完成


### 執行結果解讀

這一格只是在把 09 需要的套件、路徑、preprocessing 規則和 model builder 準備好。畫面顯示 **「模型解釋環境已準備完成」**，代表後面可以用和 08 相同的流程重建模型。

這裡還沒有做 feature importance，也沒有新的模型結果，所以這一格本身不用下製程結論。它的意義是先把環境固定，避免 09 和 08 用到不同的 preprocessing 或模型設定。


## 2. 讀入 07 / 08 artifacts，先確認我解釋的是同一套模型結果

在看 feature importance 以前，我先確認幾件基本但很重要的事：

- Base / Extended dataset 的 wafer identity 和 target 要一致。
- 08 的 Nested OOF predictions 要對得回 07 的 labeled wafers。
- 10 個 outer Lots 必須完整、不能重複。
- final model、final config 和 coefficient artifact 要是同一版。

如果這裡對不上，後面再漂亮的 explainability 都沒有意義，所以這一步先把資料 lineage 鎖住。

In [2]:
base_path = PROCESSED_DATA_DIR / "07_modeling_dataset_base.csv"
extended_path = PROCESSED_DATA_DIR / "07_modeling_dataset_extended.csv"
metadata_path = PROCESSED_DATA_DIR / "07_engineered_feature_metadata.csv"
oof_path = PROCESSED_DATA_DIR / "08_nested_oof_predictions.csv"
outer_path = PROCESSED_DATA_DIR / "08_nested_outer_fold_results.csv"
coef_path = PROCESSED_DATA_DIR / "08_final_model_coefficients.csv"
config_path = PROCESSED_DATA_DIR / "08_final_model_config.json"
model_path = MODEL_DIR / "08_final_virtual_metrology_model.joblib"

required = [base_path, extended_path, metadata_path, oof_path, outer_path, coef_path, config_path, model_path]
missing = [x for x in required if not x.exists()]
if missing:
    raise FileNotFoundError("缺少必要 artifact：" + ", ".join(str(x) for x in missing))

base_df = pd.read_csv(base_path)
extended_df = pd.read_csv(extended_path)
feature_metadata = pd.read_csv(metadata_path)
nested_oof = pd.read_csv(oof_path)
outer_results = pd.read_csv(outer_path)
final_coefficients = pd.read_csv(coef_path)
with open(config_path, "r", encoding="utf-8") as f:
    final_config = json.load(f)
final_estimator = joblib.load(model_path)

feature_set_to_df = {"Base": base_df, "Extended": extended_df}
if final_config["feature_set"] not in feature_set_to_df:
    raise ValueError("Final feature_set 不在 Base / Extended")
final_df = feature_set_to_df[final_config["feature_set"]]
feature_order = list(final_config["input_predictors_in_order"])

identity = ["experiment_key", "lot_number", "wafer_number", "mean_si_etch"]
for name, df in {"Base": base_df, "Extended": extended_df}.items():
    if df["experiment_key"].duplicated().any():
        raise ValueError(f"{name} 有重複 experiment_key")

base_id = base_df[identity].sort_values("experiment_key").reset_index(drop=True)
ext_id = extended_df[identity].sort_values("experiment_key").reset_index(drop=True)
if not base_id["experiment_key"].equals(ext_id["experiment_key"]):
    raise ValueError("Base / Extended experiment_key 不一致")
if not np.allclose(base_id["mean_si_etch"], ext_id["mean_si_etch"], rtol=0, atol=1e-12):
    raise ValueError("Base / Extended target 不一致")

merged = final_df[identity].merge(
    nested_oof[["experiment_key","lot_number","wafer_number","actual_mean_si_etch"]],
    on="experiment_key", how="outer", suffixes=("_07","_08"), indicator=True, validate="one_to_one"
)
if not (merged["_merge"] == "both").all():
    raise ValueError("07 / 08 OOF wafer identity 不一致")
if not np.allclose(merged["mean_si_etch"], merged["actual_mean_si_etch"], rtol=0, atol=1e-12):
    raise ValueError("07 target 與 08 OOF actual 不一致")
if outer_results["test_lot"].duplicated().any() or set(outer_results["test_lot"]) != set(final_df["lot_number"]):
    raise ValueError("Outer-fold test_lot 不完整或重複")

X_final = final_df[feature_order].to_numpy(dtype=float)
y = final_df["mean_si_etch"].to_numpy(dtype=float)
groups = final_df["lot_number"].to_numpy(dtype=int)
full_pred = np.asarray(final_estimator.predict(X_final)).reshape(-1)
if len(full_pred) != len(final_df):
    raise ValueError("Final model prediction length 不一致")

linear_available = bool(final_config.get("linear_coefficients_available", False))
if linear_available:
    kept = list(final_config["predictors_after_variance_filter"])
    if set(final_coefficients["feature"]) != set(kept):
        raise ValueError("08 coefficient artifact 與 final kept predictors 不一致")
else:
    if len(final_coefficients) != 0:
        raise ValueError("Non-linear final model 不應輸出直接 linear coefficients")

pooled_rmse = float(np.sqrt(np.mean(nested_oof["nested_error"] ** 2)))
pooled_mae = float(np.mean(nested_oof["abs_nested_error"]))
pooled_r2 = float(r2_score(nested_oof["actual_mean_si_etch"], nested_oof["nested_process_pred"]))

print("Final configuration：", final_config["feature_set"], "+", final_config["model"], final_config["params"])
print("是否有可直接解讀的線性係數：", linear_available)
print("Nested OOF RMSE：", round(pooled_rmse, 6))
print("Nested OOF MAE：", round(pooled_mae, 6))
print("Nested OOF R²：", round(pooled_r2, 6))


Final configuration： Extended + ElasticNet {'alpha': 0.01, 'l1_ratio': 0.1}
是否有可直接解讀的線性係數： True
Nested OOF RMSE： 0.129633
Nested OOF MAE： 0.097232
Nested OOF R²： 0.894771


### 結果解讀

這一格確認 09 讀到的 final configuration 是 **Extended + ElasticNet（alpha=0.01、l1_ratio=0.1）**，而且這個模型有可以直接解讀的線性係數。

同時，08 的 Nested OOF 結果也成功對回來：

- RMSE = **0.129633 µm**
- MAE = **0.097232 µm**
- R² = **0.894771**

這幾個數字不是 09 重新訓練後得到的新成績，而是 08 原本那批 held-out-Lot predictions。這一步最重要的意思是：**後面所有 explainability 和 error analysis 都是在解釋同一套 08 模型結果**，沒有另外換一組比較好看的預測。


## 3. 先看 Final ElasticNet 的整體係數分布

Final model 是線性模型時，因為 model 前面有 `StandardScaler`，所以 standardized coefficient 的大小可以拿來比較模型對不同 features 的權重。

這裡會看兩層：

- **單一 engineered feature**：哪個 feature 的 absolute coefficient 最大。
- **Sensor group**：把同一個 sensor 衍生出的 mean / std / robust width / early-late 等 features 放在一起，看模型的權重主要集中在哪些 sensors。

係數正負只代表在目前模型裡的預測方向，而且 predictors 彼此可能高度相關，所以這裡看的是**模型依賴**，不是製程因果順位。

In [3]:
meta_map = feature_metadata.set_index("engineered_feature")[["feature_family","sensor_short","raw_sensor"]]

if linear_available:
    global_explanation = final_coefficients.copy()
    global_explanation = global_explanation.join(meta_map, on="feature")
    global_explanation["is_nonzero"] = global_explanation["abs_coefficient"] > 1e-12
    global_explanation = global_explanation.sort_values("abs_coefficient", ascending=False).reset_index(drop=True)

    sensor_coefficient_summary = (
        global_explanation.groupby("sensor_short", as_index=False)
        .agg(
            n_features=("feature", "size"),
            n_nonzero=("is_nonzero", "sum"),
            abs_coefficient_sum=("abs_coefficient", "sum"),
            max_abs_coefficient=("abs_coefficient", "max"),
        )
        .sort_values("abs_coefficient_sum", ascending=False).reset_index(drop=True)
    )
    family_coefficient_summary = (
        global_explanation.groupby("feature_family", as_index=False)
        .agg(
            n_features=("feature", "size"),
            n_nonzero=("is_nonzero", "sum"),
            abs_coefficient_sum=("abs_coefficient", "sum"),
            max_abs_coefficient=("abs_coefficient", "max"),
        )
        .sort_values("abs_coefficient_sum", ascending=False).reset_index(drop=True)
    )

    print("權重最大的 engineered features：")
    display(global_explanation.head(15).round(5))
    print("把同一 sensor 的 features 合併後：")
    display(sensor_coefficient_summary.head(12).round(5))

    top_feature = global_explanation.iloc[0]
    top_sensor = sensor_coefficient_summary.iloc[0]
else:
    global_explanation = pd.DataFrame(columns=["feature","coefficient_standardized","abs_coefficient","feature_family","sensor_short","raw_sensor","is_nonzero"])
    sensor_coefficient_summary = pd.DataFrame(columns=["sensor_short","n_features","n_nonzero","abs_coefficient_sum","max_abs_coefficient"])
    family_coefficient_summary = pd.DataFrame(columns=["feature_family","n_features","n_nonzero","abs_coefficient_sum","max_abs_coefficient"])


權重最大的 engineered features：


,feature,coefficient_standardized,abs_coefficient,feature_family,sensor_short,raw_sensor,is_nonzero
0,robust_width__PlatenRFTuningCapacitor,0.15959,0.15959,robust_width,PlatenRFTuningCapacitor,Stat3_Etch_MV_PlatenRFTuningCapacitor,True
1,mean__PlatenRFTuningCapacitor,-0.10487,0.10487,mean,PlatenRFTuningCapacitor,Stat3_Etch_MV_PlatenRFTuningCapacitor,True
2,main_active_duration,0.09182,0.09182,timing,MainActiveWindow,Derived from SourceRFLoadPower threshold window,True
3,robust_width__PlatenRFReflectedPower,-0.09083,0.09083,robust_width,PlatenRFReflectedPower,Stat3_Etch_MV_PlatenRFReflectedPower,True
4,std__ForeLinePressure,-0.07588,0.07588,std,ForeLinePressure,Stat3_Etch_MV_ForeLinePressure,True
5,mean__Heater2Temp,0.05977,0.05977,mean,Heater2Temp,Stat3_Etch_MV_Heater2Temp,True
6,std__SourceRFReflectedPower,-0.05684,0.05684,std,SourceRFReflectedPower,Stat3_Etch_MV_SourceRFReflectedPower,True
7,mean__SourceRFLoadPower,0.04794,0.04794,mean,SourceRFLoadPower,Stat3_Etch_MV_SourceRFLoadPower,True
8,std__SourceRFPeakToPeak,0.04177,0.04177,std,SourceRFPeakToPeak,Stat3_Etch_MV_SourceRFPeakToPeak,True
9,late_minus_early__PlatenRFLoadCapacitor,-0.03816,0.03816,late_minus_early,PlatenRFLoadCapacitor,Stat3_Etch_MV_PlatenRFLoadCapacitor,True


把同一 sensor 的 features 合併後：


,sensor_short,n_features,n_nonzero,abs_coefficient_sum,max_abs_coefficient
0,PlatenRFTuningCapacitor,4,3,0.26939,0.15959
1,PlatenRFReflectedPower,4,3,0.11761,0.09083
2,ForeLinePressure,4,4,0.10781,0.07588
3,MainActiveWindow,1,1,0.09182,0.09182
4,SourceRFReflectedPower,4,2,0.07620,0.05684
5,Heater2Temp,4,3,0.07293,0.05977
6,SourceRFPeakToPeak,4,2,0.05505,0.04177
7,PlatenRFPeakToPeak,4,3,0.05198,0.02665
8,SourceRFLoadPower,4,2,0.05103,0.04794
9,Gas7Flow,4,3,0.04582,0.02355


### 結果解讀

單一 engineered feature 裡，absolute standardized coefficient 最大的是 **`robust_width__PlatenRFTuningCapacitor`**，|coefficient| = **0.15959**。

如果把同一個 sensor 衍生出的 features 合併來看，總權重最高的也是 **PlatenRFTuningCapacitor**，absolute coefficient sum = **0.26939**。

我會把這個結果解讀成：**目前 final ElasticNet 在做預測時，對 PlatenRFTuningCapacitor 這組資訊的依賴比較明顯。** 不過這裡只能講模型依賴，不能直接說它就是最重要的物理製程參數。因為 predictors 之間彼此相關，權重可能會互相分攤；係數正負也只是模型中的預測方向，不等於真實因果效果。


## 4. 固定 Final Configuration，檢查係數對不同 Lot 是否穩定

上一節只看 full-data final model，還不能知道係數是不是被某一個 Lot 撐起來的。

所以這裡把 08 最後選到的 configuration 固定不變，做 10 次 Leave-One-Lot-Out refit：每次拿掉一個 Lot，再看每個 feature 是否還是 non-zero、方向有沒有翻轉。

這一節的目的只有**stability diagnostic**。因為 configuration 已經先用完整流程選過，所以這裡算出的 fixed-configuration OOF RMSE 不是新的 unbiased performance，主模型表現仍然要看 08 的 Nested OOF。

In [4]:
logo = LeaveOneGroupOut()
n_folds = len(np.unique(groups))

if linear_available:
    coef_records = []
    fixed_oof = np.full(len(y), np.nan)
    for fold, (tr, te) in enumerate(logo.split(X_final, y, groups), start=1):
        est = clone(final_estimator)
        est.fit(X_final[tr], y[tr])
        fixed_oof[te] = np.asarray(est.predict(X_final[te])).reshape(-1)
        mask = est.named_steps["variance"].get_support()
        kept_features = np.asarray(feature_order)[mask]
        coef = np.asarray(est.named_steps["model"].coef_, dtype=float).reshape(-1)
        if len(coef) != len(kept_features):
            raise ValueError("Fold coefficient count 不一致")
        cmap = dict(zip(kept_features, coef))
        test_lot = int(np.unique(groups[te])[0])
        for f in feature_order:
            coef_records.append({"fold":fold,"test_lot":test_lot,"feature":f,"coefficient":float(cmap.get(f,0.0))})

    coefficient_by_fold = pd.DataFrame(coef_records)
    def sign_consistency(s):
        v=np.asarray(s,dtype=float); nz=v[np.abs(v)>1e-12]
        if len(nz)==0: return np.nan
        return float(max(np.mean(nz>0),np.mean(nz<0)))
    coefficient_stability = (
        coefficient_by_fold.groupby("feature",as_index=False)
        .agg(
            nonzero_folds=("coefficient",lambda s:int((np.abs(s)>1e-12).sum())),
            positive_folds=("coefficient",lambda s:int((s>1e-12).sum())),
            negative_folds=("coefficient",lambda s:int((s<-1e-12).sum())),
            median_coefficient=("coefficient","median"),
            median_abs_coefficient=("coefficient",lambda s:float(np.median(np.abs(s)))),
            coefficient_min=("coefficient","min"),
            coefficient_max=("coefficient","max"),
            sign_consistency=("coefficient",sign_consistency),
        )
        .merge(pd.DataFrame({"feature":feature_order}),on="feature",how="right")
        .merge(global_explanation[["feature","coefficient_standardized","abs_coefficient"]],on="feature",how="left")
    )
    coefficient_stability[["coefficient_standardized","abs_coefficient"]] = coefficient_stability[["coefficient_standardized","abs_coefficient"]].fillna(0.0)
    coefficient_stability = coefficient_stability.sort_values("abs_coefficient",ascending=False).reset_index(drop=True)

    stable_all = int(((coefficient_stability["nonzero_folds"]==n_folds)&(coefficient_stability["sign_consistency"]==1.0)).sum())
    min80=int(np.ceil(0.8*n_folds))
    stable80=int(((coefficient_stability["nonzero_folds"]>=min80)&(coefficient_stability["sign_consistency"]>=0.8)).sum())
    fixed_rmse=float(np.sqrt(np.mean((fixed_oof-y)**2)))

    display(coefficient_stability.head(20).round(5))
else:
    coefficient_by_fold = pd.DataFrame(columns=["fold","test_lot","feature","coefficient"])
    coefficient_stability = pd.DataFrame(columns=["feature","nonzero_folds","positive_folds","negative_folds","median_coefficient","median_abs_coefficient","coefficient_min","coefficient_max","sign_consistency","coefficient_standardized","abs_coefficient"])


,feature,nonzero_folds,positive_folds,negative_folds,median_coefficient,median_abs_coefficient,coefficient_min,coefficient_max,sign_consistency,coefficient_standardized,abs_coefficient
0,robust_width__PlatenRFTuningCapacitor,10,10,0,0.15538,0.15538,0.12203,0.16963,1.0,0.15959,0.15959
1,mean__PlatenRFTuningCapacitor,10,0,10,-0.10122,0.10122,-0.12291,-0.08279,1.0,-0.10487,0.10487
2,main_active_duration,10,10,0,0.09317,0.09317,0.06156,0.10213,1.0,0.09182,0.09182
3,robust_width__PlatenRFReflectedPower,10,0,10,-0.09098,0.09098,-0.10354,-0.06386,1.0,-0.09083,0.09083
4,std__ForeLinePressure,10,0,10,-0.07436,0.07436,-0.09270,-0.04145,1.0,-0.07588,0.07588
5,mean__Heater2Temp,10,10,0,0.06010,0.06010,0.03175,0.07410,1.0,0.05977,0.05977
6,std__SourceRFReflectedPower,10,0,10,-0.05641,0.05641,-0.06821,-0.01852,1.0,-0.05684,0.05684
7,mean__SourceRFLoadPower,10,10,0,0.05001,0.05001,0.02586,0.06329,1.0,0.04794,0.04794
8,std__SourceRFPeakToPeak,10,10,0,0.03925,0.03925,0.02870,0.05393,1.0,0.04177,0.04177
9,late_minus_early__PlatenRFLoadCapacitor,10,0,10,-0.03445,0.03445,-0.05350,-0.02160,1.0,-0.03816,0.03816


### 結果解讀

固定同一個 final configuration 做 10 次 Leave-One-Lot-Out refit 後，有 **27 個 predictors** 在 10/10 folds 都維持 non-zero，而且方向完全一致；另外有 **40 個 predictors** 至少在 8/10 folds 有權重，方向一致度也達到 80% 以上。

這表示主要的 coefficient pattern 並不是只靠某一個 Lot 才出現，至少在目前這 10 個 Lots 裡有一定穩定性。

這一格另外得到 fixed-configuration OOF RMSE 約 **0.1112 µm**。這個數字看起來比正式 Nested OOF RMSE 低，但**不能拿它當新的正式模型成績**，因為這個 configuration 是在完整 model-selection 流程之後才固定下來的。這裡真正要看的只是「係數方向穩不穩」，不是拿 0.1112 去取代 08 的 0.129633。


## 5. 用真正的 Nested Outer Models 做 Sensor-Grouped Permutation

係數只能解釋線性模型，而且同一個 sensor 常常會被拆成好幾個 engineered features。為了更直接看 held-out Lot prediction 對哪個 sensor group 最敏感，這裡會重建 08 當時的 10 個 outer models。

做法是：在每個 held-out Lot 裡，把同一個 sensor 對應的所有 engineered features 一起打亂，再看 pooled RMSE 增加多少。

如果打亂某個 sensor 後 RMSE 明顯變差，代表模型在跨 Lot 預測時確實有用到這組 wafer-specific information。但 permutation importance 一樣只是 predictive dependence，尤其 sensors 彼此相關時，不能直接當成因果重要性。

In [5]:
metadata_lookup = feature_metadata.set_index("engineered_feature")
all_sensors = sorted(feature_metadata["sensor_short"].dropna().unique().tolist())

# Rebuild every exact outer model and first verify that its predictions reproduce 08 OOF.
outer_cache = []
reconstructed_oof = np.full(len(nested_oof), np.nan)
key_to_oof_index = {k:i for i,k in enumerate(nested_oof["experiment_key"])}

for _, row in outer_results.sort_values("test_lot").iterrows():
    test_lot=int(row["test_lot"]); fs=row["feature_set"]; model=row["model"]; params=json.loads(row["params"])
    df=feature_set_to_df[fs]
    metadata_cols=["experiment_key","lot_number","wafer_number"]
    target_cols=["mean_si_etch","cv_si_etch_pct","std_si_etch","range_si_etch","p90_width_si_etch"]
    predictors=[c for c in df.columns if c not in metadata_cols+target_cols]
    X=df[predictors].to_numpy(dtype=float); yy=df["mean_si_etch"].to_numpy(dtype=float); gg=df["lot_number"].to_numpy(dtype=int)
    tr=np.where(gg!=test_lot)[0]; te=np.where(gg==test_lot)[0]
    est=build_estimator(model,params); est.fit(X[tr],yy[tr]); pred=np.asarray(est.predict(X[te])).reshape(-1)
    keys=df.iloc[te]["experiment_key"].tolist()
    for k,pr in zip(keys,pred): reconstructed_oof[key_to_oof_index[k]]=pr
    outer_cache.append({"test_lot":test_lot,"feature_set":fs,"model":model,"params":params,"df":df,"predictors":predictors,"X":X,"y":yy,"groups":gg,"train_idx":tr,"test_idx":te,"estimator":est})

if np.isnan(reconstructed_oof).any():
    raise ValueError("Outer reconstruction 有缺值")
expected = nested_oof["nested_process_pred"].to_numpy(dtype=float)
max_reconstruction_diff=float(np.max(np.abs(reconstructed_oof-expected)))
if max_reconstruction_diff > 1e-9:
    raise ValueError(f"Outer model reconstruction 無法重現 08 OOF，max diff={max_reconstruction_diff}")

rng=np.random.default_rng(42)
n_repeats=30
perm_records=[]
base_rmse=pooled_rmse

for sensor in all_sensors:
    repeat_deltas=[]
    for repeat in range(n_repeats):
        perm_pred=np.full(len(nested_oof),np.nan)
        for cache in outer_cache:
            te=cache["test_idx"]; Xtest=cache["X"][te].copy(); predictors=cache["predictors"]
            sensor_features = feature_metadata.loc[feature_metadata["sensor_short"]==sensor,"engineered_feature"].tolist()
            col_idx=[predictors.index(f) for f in sensor_features if f in predictors]
            if col_idx and len(te)>1:
                order=rng.permutation(len(te))
                Xtest[:,col_idx]=Xtest[order][:,col_idx]
            pred=np.asarray(cache["estimator"].predict(Xtest)).reshape(-1)
            keys=cache["df"].iloc[te]["experiment_key"].tolist()
            for k,pr in zip(keys,pred): perm_pred[key_to_oof_index[k]]=pr
        prmse=float(np.sqrt(np.mean((perm_pred-nested_oof["actual_mean_si_etch"].to_numpy(dtype=float))**2)))
        repeat_deltas.append(prmse-base_rmse)
    perm_records.append({
        "sensor":sensor,
        "n_repeats":n_repeats,
        "mean_delta_rmse":float(np.mean(repeat_deltas)),
        "std_delta_rmse":float(np.std(repeat_deltas,ddof=1)),
        "positive_repeat_fraction":float(np.mean(np.asarray(repeat_deltas)>0)),
    })

sensor_permutation = pd.DataFrame(perm_records).sort_values("mean_delta_rmse",ascending=False).reset_index(drop=True)
print("重建 outer models 後，和 08 Nested OOF 的最大 prediction 差異：", max_reconstruction_diff)
display(sensor_permutation.head(15).round(5))

top_perm=sensor_permutation.iloc[0]
second_perm=sensor_permutation.iloc[1] if len(sensor_permutation) > 1 else None
perm_ratio=(top_perm["mean_delta_rmse"] / second_perm["mean_delta_rmse"]) if second_perm is not None and second_perm["mean_delta_rmse"] > 0 else np.nan


重建 outer models 後，和 08 Nested OOF 的最大 prediction 差異： 7.105427357601002e-15


,sensor,n_repeats,mean_delta_rmse,std_delta_rmse,positive_repeat_fraction
0,PlatenRFTuningCapacitor,30,0.24996,0.02185,1.00000
1,MainActiveWindow,30,0.05900,0.01242,1.00000
2,ForeLinePressure,30,0.03235,0.00597,1.00000
3,SourceRFReflectedPower,30,0.02489,0.00461,1.00000
4,PlatenRFReflectedPower,30,0.01596,0.00366,1.00000
5,SourceRFLoadPower,30,0.01398,0.00440,1.00000
6,SourceRFPeakToPeak,30,0.00924,0.00344,1.00000
7,PlatenRFLoadCapacitor,30,0.00333,0.00187,0.96667
8,PlatenRFLoadPower,30,0.00200,0.00094,0.96667
9,Gas2Flow,30,0.00151,0.00131,0.83333


### 結果解讀

先看 lineage check：重建 10 個 outer models 後，和 08 Nested OOF predictions 的最大差只有 **7.11 × 10⁻¹⁵ µm**。這已經是浮點數精度等級，可以視為同一批 prediction，代表 09 沒有重建錯模型。

接著看 sensor-group permutation。打亂 **PlatenRFTuningCapacitor** 之後，RMSE 平均增加 **0.24996 µm**，而且 30 次 repeats 裡 **100% 都讓 RMSE 變差**。第二名 MainActiveWindow 的平均 ΔRMSE 約 **0.05900 µm**，所以第一名的影響大約是第二名的 4 倍以上。

這和前面的 coefficient 結果互相呼應：兩種不同方法都顯示模型很依賴 PlatenRFTuningCapacitor 這組資訊。這讓「模型依賴這組 sensor」的證據比較一致，但仍然不能把它寫成「這個 sensor 就是 etch 結果的根因」。


## 6. Nested OOF Error：先找出最難泛化的 Lot 和 wafer

接下來不再重跑模型，只整理 08 已經產生的 Nested OOF errors。

我會先比較每個 Lot 的 RMSE / MAE / max absolute error，再列出單片誤差最大的 wafers。這樣可以分清楚：是只有某一片特別難預測，還是某一整個 Lot 對模型來說都比較陌生。

In [6]:
error_by_lot = (
    nested_oof.groupby("lot_number",as_index=False)
    .agg(
        n_wafers=("experiment_key","size"),
        rmse=("nested_error",lambda s:float(np.sqrt(np.mean(np.asarray(s,dtype=float)**2)))),
        mae=("abs_nested_error","mean"),
        max_abs_error=("abs_nested_error","max"),
    )
    .sort_values("rmse",ascending=False).reset_index(drop=True)
)
top_errors = nested_oof.sort_values("abs_nested_error",ascending=False).head(10).reset_index(drop=True)

display(error_by_lot.round(5))
print("單片 absolute error 最大的 wafers：")
display(top_errors[["experiment_key","lot_number","wafer_number","actual_mean_si_etch","nested_process_pred","nested_error","abs_nested_error"]].round(5))

worst_lot_row = error_by_lot.iloc[0]
worst_lot_number = int(worst_lot_row["lot_number"])
worst_wafer = top_errors.iloc[0]
worst_lot_top10_count = int((top_errors["lot_number"].astype(int) == worst_lot_number).sum())


,lot_number,n_wafers,rmse,mae,max_abs_error
0,8,10,0.22384,0.14756,0.57555
1,3,10,0.14304,0.12877,0.22756
2,5,10,0.13771,0.11333,0.27007
3,7,6,0.11837,0.09339,0.20857
4,6,10,0.11752,0.08241,0.27963
5,4,10,0.11432,0.09827,0.21846
6,2,10,0.10996,0.09629,0.16586
7,10,4,0.08315,0.07746,0.10814
8,9,9,0.06980,0.05825,0.12378
9,1,9,0.06516,0.05508,0.11583


單片 absolute error 最大的 wafers：


,experiment_key,lot_number,wafer_number,actual_mean_si_etch,nested_process_pred,nested_error,abs_nested_error
0,2024-08-07_10,8,10,43.21752,43.79308,0.57555,0.57555
1,2024-08-01_01,6,1,43.75222,44.03186,0.27963,0.27963
2,2024-07-19_01,5,1,44.31420,44.04413,-0.27007,0.27007
3,2024-08-07_08,8,8,43.69908,43.95658,0.25749,0.25749
4,2024-07-09_03,3,3,44.26601,44.03845,-0.22756,0.22756
5,2024-08-07_09,8,9,43.47381,43.69984,0.22602,0.22602
6,2024-07-11_04,4,4,43.99857,43.78011,-0.21846,0.21846
7,2024-08-05_01,7,1,43.83111,44.03968,0.20857,0.20857
8,2024-08-07_01,8,1,43.74446,43.94651,0.20205,0.20205
9,2024-07-19_08,5,8,43.17213,43.37239,0.20026,0.20026


### 結果解讀

Lot-level 結果裡，RMSE 最高的是 **Lot 8 = 0.22384 µm**；最低的是 Lot 1 = 0.06516 µm。這代表不同 Lots 的泛化難度差很多，不能只看 pooled RMSE 就認為每個 Lot 都表現一樣。

單片 wafer 裡，absolute OOF error 最大的是 **2024-08-07_10**，誤差 **0.57555 µm**，而且它也屬於 Lot 8。前 10 大單片 error 裡，**Lot 8 一共有 4 片**。

所以 Lot 8 的問題不像只有一片 wafer 偶然失準，而是整個 Lot 對「只用其他 Lots 訓練」的模型都比較難預測。這裡只能說明 **generalization difficulty**，還不能只靠 OOF error 就判定設備異常或 wafer defect。


## 7. 往下追 Worst Wafer：當時的 Outer Model 到底受哪些 features 影響？

找到最大 OOF error 之後，這裡會回到**當時真的沒看過該 Lot 的 outer model**，而不是拿 full-data model 事後解釋。

對這片 wafer，我把每個 feature 一次一個 reset 成 outer-training mean，再看 prediction 會改多少。若 outer model 是 Ridge / ElasticNet，還會另外計算 exact standardized linear contribution。

這樣可以回答「模型對這片 wafer 的哪些偏離最敏感」，但不能直接說哪一個 feature 就是製程異常的根因。

In [7]:
worst = top_errors.iloc[0]
worst_key = str(worst["experiment_key"])
worst_wafer_lot_number = int(worst["lot_number"])
cache = next(x for x in outer_cache if x["test_lot"] == worst_wafer_lot_number)
df=cache["df"]; predictors=cache["predictors"]; X=cache["X"]; tr=cache["train_idx"]; te=cache["test_idx"]; est=cache["estimator"]
row_idx=int(df.index[df["experiment_key"]==worst_key][0])
# DataFrame index is RangeIndex in 07 artifacts; validate membership in held-out fold.
if row_idx not in set(te.tolist()):
    raise ValueError("Worst wafer 不在 reconstructed outer test fold")
xrow=X[[row_idx]].copy(); pred0=float(np.asarray(est.predict(xrow)).reshape(-1)[0])
expected_pred=float(worst["nested_process_pred"])
if not np.isclose(pred0,expected_pred,rtol=0,atol=1e-9):
    raise ValueError("Worst-wafer reconstructed prediction 對不上 08")

train_means=np.mean(X[tr],axis=0)
local=[]
for j,f in enumerate(predictors):
    xreset=xrow.copy(); xreset[0,j]=train_means[j]
    pred_reset=float(np.asarray(est.predict(xreset)).reshape(-1)[0])
    local.append({"feature":f,"local_reset_sensitivity":pred0-pred_reset,"abs_local_reset_sensitivity":abs(pred0-pred_reset)})
local_df=pd.DataFrame(local)

mask=est.named_steps["variance"].get_support(); kept=np.asarray(predictors)[mask]
z=est.named_steps["scale"].transform(est.named_steps["variance"].transform(xrow)).reshape(-1)
shift_df=pd.DataFrame({"feature":kept,"standardized_shift":z,"abs_standardized_shift":np.abs(z)})
local_df=local_df.merge(shift_df,on="feature",how="left")
local_df=local_df.join(meta_map,on="feature")

outer_linear = cache["model"] in {"Ridge","ElasticNet"}
local_df["exact_linear_contribution"] = np.nan
if outer_linear:
    coef=np.asarray(est.named_steps["model"].coef_,dtype=float).reshape(-1)
    if len(coef)!=len(kept): raise ValueError("Outer linear coefficient length 不一致")
    contrib=z*coef
    cmap=dict(zip(kept,contrib))
    local_df["exact_linear_contribution"] = local_df["feature"].map(cmap)
    intercept=float(np.asarray(est.named_steps["model"].intercept_).reshape(-1)[0])
    reconstructed=intercept+float(np.sum(contrib))
    if not np.isclose(reconstructed,pred0,rtol=0,atol=1e-8):
        raise ValueError("Exact linear contribution decomposition 無法重現 prediction")

local_df=local_df.sort_values("abs_local_reset_sensitivity",ascending=False).reset_index(drop=True)
print("最大 OOF error wafer：", worst_key, "| Lot", worst_wafer_lot_number, "| 當時 outer model", cache["feature_set"], "+", cache["model"], cache["params"])
print("實際值：",round(float(worst["actual_mean_si_etch"]),6),"預測值：",round(pred0,6),"誤差：",round(float(worst["nested_error"]),6))
display(local_df.head(15).round(5))

top_local=local_df.iloc[0]
if outer_linear:
    top_up = local_df.sort_values("exact_linear_contribution", ascending=False).iloc[0]
    top_down = local_df.sort_values("exact_linear_contribution", ascending=True).iloc[0]
else:
    top_up = local_df.sort_values("local_reset_sensitivity", ascending=False).iloc[0]
    top_down = local_df.sort_values("local_reset_sensitivity", ascending=True).iloc[0]


最大 OOF error wafer： 2024-08-07_10 | Lot 8 | 當時 outer model Base + ElasticNet {'alpha': 0.01, 'l1_ratio': 0.5}
實際值： 43.217525 預測值： 43.793077 誤差： 0.575552


,feature,local_reset_sensitivity,abs_local_reset_sensitivity,standardized_shift,abs_standardized_shift,feature_family,sensor_short,raw_sensor,exact_linear_contribution
0,std__SourceRFReflectedPower,-0.18161,0.18161,1.96562,1.96562,std,SourceRFReflectedPower,Stat3_Etch_MV_SourceRFReflectedPower,-0.18161
1,std__ForeLinePressure,-0.09692,0.09692,1.60150,1.60150,std,ForeLinePressure,Stat3_Etch_MV_ForeLinePressure,-0.09692
2,mean__Heater2Temp,0.08850,0.08850,0.95620,0.95620,mean,Heater2Temp,Stat3_Etch_MV_Heater2Temp,0.08850
3,late_minus_early__Gas7Flow,0.08735,0.08735,3.70050,3.70050,late_minus_early,Gas7Flow,Stat3_Etch_MV_Gas7Flow,0.08735
4,std__PlatenDcBias,0.04775,0.04775,1.73289,1.73289,std,PlatenDcBias,Stat3_Etch_MV_PlatenDcBias,0.04775
5,std__Heater2Temp,0.04234,0.04234,-2.79556,2.79556,std,Heater2Temp,Stat3_Etch_MV_Heater2Temp,0.04234
6,mean__SourceRFLoadPower,0.03436,0.03436,0.77736,0.77736,mean,SourceRFLoadPower,Stat3_Etch_MV_SourceRFLoadPower,0.03436
7,main_active_duration,0.02692,0.02692,0.22499,0.22499,timing,MainActiveWindow,Derived from SourceRFLoadPower threshold window,0.02692
8,std__Gas2Flow,0.02607,0.02607,0.97653,0.97653,std,Gas2Flow,Stat3_Etch_MV_Gas2Flow,0.02607
9,mean__SourceRFReflectedPower,-0.02299,0.02299,1.33148,1.33148,mean,SourceRFReflectedPower,Stat3_Etch_MV_SourceRFReflectedPower,-0.02299


### 結果解讀

最大 OOF error wafer 是 **2024-08-07_10（Lot 8）**。實際 Mean Si Etch = **43.21752 µm**，模型預測 = **43.79308 µm**，所以 prediction − actual = **+0.57555 µm**，也就是模型高估。

把每個 feature 一次一個 reset 回 outer-training mean 後，absolute sensitivity 最大的是 **`std__SourceRFReflectedPower` = -0.18161 µm**。這個值是負的，意思不是它造成高估，反而表示這個 feature 在目前 wafer 上有把 prediction 往下拉、抵消部分高估。

目前往上推 prediction 最明顯的是 **`mean__Heater2Temp`，約 +0.08850 µm**；往下拉最明顯的是 **`std__SourceRFReflectedPower`，約 -0.18161 µm**。

所以這片 wafer 的大誤差不是一個 feature 就能解釋，而是多個 feature 同時往不同方向作用後形成的結果。這個 local explanation 適合拿來找「模型在哪些 process directions 最敏感」，不適合直接把第一名當成製程根因。


## 8. 輸出 09 的 Explainability Artifacts

最後把 global coefficients、coefficient stability、sensor-group permutation、Lot / wafer OOF errors，以及 worst-wafer local evidence 全部存成 CSV。

每一個檔案寫出後都會再讀回來檢查 row count 和欄位，讓 10 可以直接接著用這一版結果，不需要重新猜 final model、sensor ranking 或 worst wafer。

In [8]:
paths = {
    "global": PROCESSED_DATA_DIR / "09_final_feature_explanation.csv",
    "stability": PROCESSED_DATA_DIR / "09_coefficient_stability.csv",
    "sensor_coef": PROCESSED_DATA_DIR / "09_sensor_coefficient_summary.csv",
    "family_coef": PROCESSED_DATA_DIR / "09_family_coefficient_summary.csv",
    "sensor_perm": PROCESSED_DATA_DIR / "09_sensor_grouped_permutation_importance.csv",
    "error_lot": PROCESSED_DATA_DIR / "09_nested_error_by_lot.csv",
    "top_errors": PROCESSED_DATA_DIR / "09_top_nested_oof_errors.csv",
    "worst_local": PROCESSED_DATA_DIR / "09_worst_wafer_contributions.csv",
    "worst_shift": PROCESSED_DATA_DIR / "09_worst_wafer_feature_shift.csv",
}
artifacts = {
    paths["global"]: global_explanation,
    paths["stability"]: coefficient_stability,
    paths["sensor_coef"]: sensor_coefficient_summary,
    paths["family_coef"]: family_coefficient_summary,
    paths["sensor_perm"]: sensor_permutation,
    paths["error_lot"]: error_by_lot,
    paths["top_errors"]: top_errors,
    paths["worst_local"]: local_df,
    paths["worst_shift"]: local_df[["feature","standardized_shift","abs_standardized_shift","feature_family","sensor_short","raw_sensor"]].copy(),
}
validation=[]
for path,df in artifacts.items():
    df.to_csv(path,index=False)
    chk=pd.read_csv(path)
    if len(chk)!=len(df) or list(chk.columns)!=list(df.columns):
        raise ValueError(f"09 artifact validation failed: {path.name}")
    validation.append({"artifact":path.name,"rows":len(chk),"schema_match":True})
validation_df=pd.DataFrame(validation)
display(validation_df)


,artifact,rows,schema_match
0,09_final_feature_explanation.csv,98,True
1,09_coefficient_stability.csv,109,True
2,09_sensor_coefficient_summary.csv,26,True
3,09_family_coefficient_summary.csv,5,True
4,09_sensor_grouped_permutation_importance.csv,28,True
5,09_nested_error_by_lot.csv,10,True
6,09_top_nested_oof_errors.csv,10,True
7,09_worst_wafer_contributions.csv,82,True
8,09_worst_wafer_feature_shift.csv,82,True


### 輸出結果解讀

09 一共寫出 **9 個 CSV artifacts**，而且每一個檔案都重新讀回檢查 row count 和欄位，全部 `schema_match=True`。

這代表後面的 10 可以直接使用這一版的 coefficient、stability、permutation、Lot error、top-error wafers 和 local explanation，不需要自己再重算一份。對整個專案來說，這一步的意義是把 **08 → 09 → 10 的資料 lineage 固定下來**，減少不同 notebook 各自保存不同版本結果的風險。


## 9. 09 的整體結論

前面每一段都已經分開看過輸出結果，最後這裡只做整本 09 的統整，不再跑新的分析。


### 統整結論

把這一本的結果放在一起後，我會這樣解讀 09：

第一，09 成功對回 08 的同一批 Nested OOF predictions，正式的 held-out-Lot 成績仍然是 **RMSE 0.129633 µm、MAE 0.097232 µm、R² 0.894771**。所以這一本不是重新找一個更漂亮的模型，而是在解釋原本那一套模型。

第二，global coefficient 和 nested sensor-group permutation 都把 **PlatenRFTuningCapacitor** 指到最前面。單一最大 standardized coefficient 是 `robust_width__PlatenRFTuningCapacitor`（|coef| = **0.15959**），sensor-level coefficient sum 也是 PlatenRFTuningCapacitor 最高；真正用 nested outer models 做 permutation 時，它的平均 ΔRMSE 也最高，達到 **+0.24996 µm**。兩種方法互相呼應，讓我比較有把握說：**目前模型確實很依賴這組 sensor information。**

第三，這個 dependence 不是只由單一 Lot 撐起來。固定 final configuration 重新做 Leave-One-Lot-Out refit 時，有 **27 個 predictors** 在 10/10 folds 都維持 non-zero 而且方向一致，表示主要 coefficient pattern 在目前 10 個 Lots 裡有一定穩定性。不過 fixed-configuration OOF RMSE 約 0.1112 µm 只能拿來做 stability diagnostic，不能取代正式 Nested OOF 成績。

第四，真正比較明顯的風險是在 **Lot-level generalization**。Lot 8 的 RMSE 最高，達 **0.22384 µm**；前 10 大單片 OOF errors 裡，有 **4 片**來自 Lot 8。最大單片 error 是 **2024-08-07_10 = 0.57555 µm**。這表示模型在 Lot 8 上不是只有一片偶然預測不好，而是整個 Lot 都比較難泛化。

最後，worst-wafer local explanation 也顯示，2024-08-07_10 的高估不是單一 feature 可以解釋。`mean__Heater2Temp` 會把 prediction 往上推，`std__SourceRFReflectedPower` 則明顯往下拉，最後是多個正負影響一起疊加。

**所以 09 最後的重點不是找出一個「最重要製程參數」，而是確認模型的 predictive dependence 有一定跨 Lot 穩定性，同時也看出 Lot 8 是目前最明顯的泛化弱點。** 這些結果可以幫忙決定後續要優先檢查哪些 sensor patterns，但不能直接當成製程因果證明。下一本 10 再把這些 model evidence 和 SPC、residual anomaly、Cross-Lot OOD 放在一起比較。
